# `agents` — Deep Agent User Guide

Orchestrates three specialised sub-agents over all **8 webai tools** via a single `build_webai_agent()` factory.

```
agents/
│
├── build_webai_agent(model, openai_api_key, tavily_api_key, *, openai_model)
│         └── returns a compiled Deep Agents graph
│
├── tools/
│   ├── make_web_search_tools(searcher)    → [web_search]
│   ├── make_ticker_tools(researcher)      → [research_ticker, research_portfolio, research_sector]
│   └── make_news_tools(researcher)        → [fetch_breaking_news, fetch_macro_news,
│                                             fetch_catalyst_news, fetch_daily_digest]
│
└── demo.py   — interactive CLI (python -m agents.demo)
```

```
                    ┌─────────────────────────┐
                    │    Orchestrator Agent    │
                    │  (routes via task tool)  │
                    └──────┬──────┬───────┬───┘
                           │      │       │
           ┌───────────────┘      │       └───────────────┐
           ▼                      ▼                        ▼
┌────────────────────┐ ┌─────────────────────┐ ┌─────────────────────┐
│  web-search-agent  │ │   ticker-agent       │ │    news-agent       │
│  • web_search      │ │  • research_ticker   │ │  • fetch_breaking   │
│                    │ │  • research_portfolio│ │  • fetch_macro      │
│                    │ │  • research_sector   │ │  • fetch_catalyst   │
└────────────────────┘ └─────────────────────┘ │  • fetch_daily      │
                                               └─────────────────────┘
```

| Sub-agent | Tools | Needs API keys |
|---|---|---|
| `web-search-agent` | `web_search` | TAVILY_API_KEY (or OPENAI_API_KEY) |
| `ticker-agent` | `research_ticker`, `research_portfolio`, `research_sector` | TAVILY_API_KEY + OPENAI_API_KEY |
| `news-agent` | `fetch_breaking_news`, `fetch_macro_news`, `fetch_catalyst_news`, `fetch_daily_digest` | TAVILY_API_KEY + OPENAI_API_KEY |

**Prerequisites:**
- Section 2 (architecture tour) and Section 3 (tool factories) run **without** API keys.
- All other sections require `TAVILY_API_KEY` and `OPENAI_API_KEY` in `.env`.
- The orchestrator uses `gpt-4o`; synthesis uses `gpt-4o-mini` — both via `OPENAI_API_KEY`.

## Sections
1. [Setup](#1-setup)
2. [Architecture tour (no API keys)](#2-architecture-tour)
3. [Tool factories (no API keys)](#3-tool-factories)
4. [Building the agent](#4-building-the-agent)
5. [web-search-agent — `web_search`](#5-web-search-agent)
6. [ticker-agent — `research_ticker`](#6-research_ticker)
7. [ticker-agent — `research_portfolio`](#7-research_portfolio)
8. [ticker-agent — `research_sector`](#8-research_sector)
9. [news-agent — `fetch_breaking_news`](#9-fetch_breaking_news)
10. [news-agent — `fetch_macro_news`](#10-fetch_macro_news)
11. [news-agent — `fetch_catalyst_news`](#11-fetch_catalyst_news)
12. [news-agent — `fetch_daily_digest`](#12-fetch_daily_digest)
13. [Multi-turn conversation](#13-multi-turn)
14. [End-to-end pattern](#14-end-to-end)
15. [Error handling reference](#15-error-handling)

## 1 — Setup <a id="1-setup"></a>

In [ ]:
import json
import logging
import os
import uuid

from dotenv import load_dotenv

# Load .env before any webai/langchain import so env-gated behaviour
# (API key checks, LangSmith tracing) sees the correct values.
load_dotenv()

# Silence LangSmith tracing noise — not needed for this guide.
os.environ.setdefault("LANGCHAIN_TRACING_V2", "false")
os.environ.setdefault("LANGSMITH_TRACING", "false")

from agents.webai_agent import build_webai_agent
from agents.tools.web_search_tools import make_web_search_tools
from agents.tools.ticker_tools import make_ticker_tools
from agents.tools.news_tools import make_news_tools

logging.basicConfig(
    level=logging.WARNING,  # flip to DEBUG to see the full pipeline trace
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    force=True,
)

_HAS_TAVILY = bool(os.environ.get("TAVILY_API_KEY"))
_HAS_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
_HAS_KEYS   = _HAS_TAVILY and _HAS_OPENAI

print(f"TAVILY_API_KEY : {'set' if _HAS_TAVILY else 'MISSING'}")
print(f"OPENAI_API_KEY : {'set' if _HAS_OPENAI else 'MISSING'}")
print(f"Live sections  : {'enabled' if _HAS_KEYS else 'SKIPPED — set both keys in .env'}")

## 2 — Architecture tour <a id="2-architecture-tour"></a>

The agent stack has four layers, each adding capabilities on top of the one below:

| Layer | Component | What it provides |
|---|---|---|
| **webai** | `WebSearcher`, `TickerResearcher`, `NewsResearcher` | Search, research, news pipelines |
| **tool factories** | `make_*_tools(client)` | Wraps each method as a LangChain `@tool` with error isolation |
| **sub-agents** | `web-search-agent`, `ticker-agent`, `news-agent` | Ephemeral agents, each with a specialised toolset |
| **orchestrator** | `webai-orchestrator` | Routes requests via `task` tool; plans with `write_todos`; synthesises results |

### How the orchestrator routes requests

The orchestrator never calls webai directly. It decides *which sub-agent* to delegate to, sends a complete self-contained instruction, and synthesises the result.

```
User: "Research NVDA for the last 7 days"
  → Orchestrator calls task(agent="ticker-agent",
        instruction="Call research_ticker('NVDA', days_back=7) ...")
  → ticker-agent runs research_ticker, returns JSON
  → Orchestrator synthesises and answers user
```

### Sub-agent statefulness

Sub-agents are **stateless/ephemeral** — each `task` call starts a fresh context. This means:
- Instructions must be **complete and self-contained** in a single call.
- The orchestrator is responsible for threading context between calls.
- The orchestrator itself has persistent memory via `MemorySaver` + `thread_id`.

## 3 — Tool factories (no API keys) <a id="3-tool-factories"></a>

Each factory takes a pre-built webai client and returns a list of LangChain `BaseTool` objects.
Using closures keeps clients as singletons and makes the dependency explicit — no global state.

This section runs entirely without API keys by using `MagicMock` clients.

In [ ]:
from unittest.mock import MagicMock

# Build mock clients — no real API calls
mock_searcher    = MagicMock()
mock_ticker_res  = MagicMock()
mock_news_res    = MagicMock()

web_tools    = make_web_search_tools(mock_searcher)
ticker_tools = make_ticker_tools(mock_ticker_res)
news_tools   = make_news_tools(mock_news_res)

print("=== Web search tools ===")
for t in web_tools:
    print(f"  {t.name:30s} {t.description[:60]}...")

print("\n=== Ticker tools ===")
for t in ticker_tools:
    print(f"  {t.name:30s} {t.description[:60]}...")

print("\n=== News tools ===")
for t in news_tools:
    print(f"  {t.name:30s} {t.description[:60]}...")

print(f"\nTotal tools: {len(web_tools) + len(ticker_tools) + len(news_tools)}")

In [ ]:
# Inspect a tool's full argument schema — useful for debugging prompt generation
import json as _json
tool_schema = ticker_tools[0].get_input_schema().model_json_schema()
print(f"research_ticker schema:")
print(_json.dumps(tool_schema, indent=2))

## 4 — Building the agent <a id="4-building-the-agent"></a>

`build_webai_agent()` is the single entry point. It:

1. Resolves API keys from arguments or environment variables.
2. Constructs `WebSearcher`, `TickerResearcher`, and `NewsResearcher` singletons.
3. Calls each tool factory to produce the three tool lists.
4. Defines the three sub-agent configs.
5. Calls `create_deep_agent()` with a `MemorySaver` checkpointer.

| Parameter | Default | Notes |
|---|---|---|
| `model` | *(required)* | Orchestrator model string, e.g. `"claude-sonnet-4-6"` |
| `openai_api_key` | `$OPENAI_API_KEY` | Used by WebSearcher fallback + synthesis LLM |
| `tavily_api_key` | `$TAVILY_API_KEY` | Used by all three researchers for search |
| `openai_model` | `"gpt-4o-mini"` | OpenAI model for structured-output synthesis in TickerResearcher / NewsResearcher |

> **Invariant:** `build_webai_agent` never raises on valid inputs. API keys missing at construction time are silently skipped by `WebSearcher`; `NewsResearcher` raises `ValueError` at construction if Tavily key is absent.

In [ ]:
if not _HAS_KEYS:
    print("Skipping — requires TAVILY_API_KEY and OPENAI_API_KEY.")
else:
    agent = build_webai_agent(
        model="gpt-4o",            # orchestrator model (OpenAI — no ANTHROPIC_API_KEY needed)
        openai_model="gpt-4o-mini", # synthesis model for structured output
    )

    # Reuse this thread_id across all live sections so the orchestrator
    # retains conversation context throughout the guide.
    THREAD_ID = str(uuid.uuid4())
    CONFIG    = {"configurable": {"thread_id": THREAD_ID}}

    print(f"Agent built successfully")
    print(f"Thread ID : {THREAD_ID}")

In [ ]:
# Helper used throughout this guide to extract the final assistant reply.
def get_reply(result: dict) -> str:
    """Return the last assistant message content from an agent result dict."""
    for msg in reversed(result.get("messages", [])):
        role    = getattr(msg, "type", None) or (msg.get("role") if isinstance(msg, dict) else None)
        content = getattr(msg, "content", None) or (msg.get("content") if isinstance(msg, dict) else None)
        if role in ("ai", "assistant") and content:
            return content if isinstance(content, str) else str(content)
    return str(result)

print("get_reply() helper defined")

## 5 — web-search-agent — `web_search` <a id="5-web-search-agent"></a>

**When to use:** General web queries that are not specific to equity analysis or news pipelines — regulatory filings, macro commentary, company announcements, background research.

The `web_search` tool exposes two backends:

| `provider` | Returns | Best for |
|---|---|---|
| `"tavily"` (default) | Numbered list of snippets with URLs | Factual, sourced research |
| `"openai"` | One synthesised prose answer | Quick summaries |

Additional Tavily parameters:
- `topic="news"` — time-sensitive queries (uses Tavily news index)
- `days=N` — restrict to the last N days (Tavily only)

The `fallback=True` flag in the underlying `WebSearcher.search()` call means if the primary provider fails the other is tried automatically.

In [ ]:
if not _HAS_KEYS:
    print("Skipping — requires TAVILY_API_KEY and OPENAI_API_KEY.")
else:
    result = agent.invoke(
        {"messages": [{"role": "user", "content":
            "Search for recent SEC enforcement actions against hedge funds in the last 7 days "
            "using Tavily with topic=news and days=7."}]},
        config=CONFIG,
    )
    print(get_reply(result))

## 6 — ticker-agent — `research_ticker` <a id="6-research_ticker"></a>

`research_ticker` runs the full equity research pipeline for a single symbol:

1. Build anchor query from symbol + optional `company_name` + optional `earnings_date`
2. LLM query fan-out (expand → decompose → step-back) with finance-domain few-shot examples
3. Parallel Tavily search (`topic="finance"`, optional `days_back` filter)
4. Deduplicate by URL → filter to `TRUSTED_DOMAINS` (22 finance sites)
5. Synthesise `TickerResearch` via `model.with_structured_output(TickerResearch)`

**Key parameters:**
- `company_name` — improves search anchor for ambiguous tickers
- `earnings_date` — appended to query and referenced in synthesis prompt; echoed in `result.earnings_date`
- `days_back` — restricts Tavily results to the last N days via `start_date`

In [ ]:
if not _HAS_KEYS:
    print("Skipping — requires TAVILY_API_KEY and OPENAI_API_KEY.")
else:
    result = agent.invoke(
        {"messages": [{"role": "user", "content":
            "Research NVDA (Nvidia) for the last 14 days. "
            "The next earnings date is 2026-05-28."}]},
        config=CONFIG,
    )
    print(get_reply(result))

## 7 — ticker-agent — `research_portfolio` <a id="7-research_portfolio"></a>

`research_portfolio` runs `research_ticker` for multiple symbols **in parallel** inside a `ThreadPoolExecutor`. Individual ticker failures are caught, logged at `WARNING`, and excluded — the batch never aborts.

**Symbol format (comma-separated string passed to the tool):**
- Bare symbol: `"NVDA"` → searched without a company name anchor
- With company name: `"NVDA|NVIDIA"` → ticker `|` company (pipe separator)

> **Invariant:** Returns `{}` if every ticker fails. Never raises on per-ticker errors.

In [ ]:
if not _HAS_KEYS:
    print("Skipping — requires TAVILY_API_KEY and OPENAI_API_KEY.")
else:
    result = agent.invoke(
        {"messages": [{"role": "user", "content":
            "Give me a portfolio snapshot for NVDA|NVIDIA, AAPL|Apple, and MSFT|Microsoft "
            "over the last 7 days. Summarise each stock's sentiment and key catalyst."}]},
        config=CONFIG,
    )
    print(get_reply(result))

## 8 — ticker-agent — `research_sector` <a id="8-research_sector"></a>

`research_sector` targets macro-level sector content. The pipeline is identical to `research_ticker` but uses sector-specific few-shot examples and synthesises a `SectorResearch` model instead of `TickerResearch`.

**`SectorResearch` fields:** `sector`, `overall_health`, `key_trends`, `tailwinds`, `headwinds`, `valuation_note`, `leading_companies`, `macro_sensitivity`, `outlook`, `sources`, `freshness_warning`

The sector name you pass is stored verbatim in `result.sector` — safe to use as a dict key for caching (see end-to-end section).

In [ ]:
if not _HAS_KEYS:
    print("Skipping — requires TAVILY_API_KEY and OPENAI_API_KEY.")
else:
    result = agent.invoke(
        {"messages": [{"role": "user", "content":
            "How is the Semiconductors sector doing right now? "
            "Include key trends, tailwinds, headwinds, and outlook."}]},
        config=CONFIG,
    )
    print(get_reply(result))

## 9 — news-agent — `fetch_breaking_news` <a id="9-fetch_breaking_news"></a>

`fetch_breaking_news` returns `list[NewsItem]` sorted by `relevance_score` descending.

**`NewsItem` fields:** `headline`, `summary`, `sentiment`, `event_type`, `relevance_score [0–1]`, `tickers_mentioned`

The tool accepts a comma-separated ticker string. Internally it splits and upper-cases each symbol before calling `NewsResearcher.fetch_breaking_news(ticker_list, days_back=...)`.

**When to use vs `fetch_catalyst_news`:**
- `fetch_breaking_news` — broad sweep across multiple tickers, last 1–2 days
- `fetch_catalyst_news` — deep dive on a single ticker's catalysts, last 7 days

In [ ]:
if not _HAS_KEYS:
    print("Skipping — requires TAVILY_API_KEY and OPENAI_API_KEY.")
else:
    result = agent.invoke(
        {"messages": [{"role": "user", "content":
            "What's breaking today for TSLA and NVDA? "
            "Look back 2 days and list the top stories by relevance."}]},
        config=CONFIG,
    )
    print(get_reply(result))

## 10 — news-agent — `fetch_macro_news` <a id="10-fetch_macro_news"></a>

`fetch_macro_news` searches without any ticker anchor — it targets broad market and macro-economic news (Fed policy, inflation prints, GDP data, credit spreads, etc.).

Results are sorted by `relevance_score` descending, same as `fetch_breaking_news`.

**Typical use cases:**
- Morning macro brief before market open
- Checking for rate decision or CPI impact
- Understanding the macro backdrop for sector research

In [ ]:
if not _HAS_KEYS:
    print("Skipping — requires TAVILY_API_KEY and OPENAI_API_KEY.")
else:
    result = agent.invoke(
        {"messages": [{"role": "user", "content":
            "Give me a macro news brief for the last 2 days. "
            "Focus on Fed policy, inflation, and any major economic data releases."}]},
        config=CONFIG,
    )
    print(get_reply(result))

## 11 — news-agent — `fetch_catalyst_news` <a id="11-fetch_catalyst_news"></a>

`fetch_catalyst_news` does a focused deep-dive on a **single ticker** over a longer lookback (default 7 days).

**Post-filter rule:** Only `NewsItem` objects where **either**:
- the ticker appears in `tickers_mentioned`, **or**
- `relevance_score >= 0.5`

are returned. This prevents loosely related results from polluting the catalyst list.

**Typical catalysts returned:** Earnings, product launches, partnerships, guidance changes, analyst upgrades/downgrades, regulatory approvals, management changes.

In [ ]:
if not _HAS_KEYS:
    print("Skipping — requires TAVILY_API_KEY and OPENAI_API_KEY.")
else:
    result = agent.invoke(
        {"messages": [{"role": "user", "content":
            "What catalysts are driving AMD over the last 7 days? "
            "List by event type and explain the relevance of each."}]},
        config=CONFIG,
    )
    print(get_reply(result))

## 12 — news-agent — `fetch_daily_digest` <a id="12-fetch_daily_digest"></a>

`fetch_daily_digest` produces a structured `NewsDigest` — a single consolidated view of the day combining ticker news, sector moves, and macro events.

**`NewsDigest` fields:**

| Field | Type | Description |
|---|---|---|
| `top_stories` | `list[NewsItem]` | Highest-relevance stories across all inputs |
| `market_theme` | `str` | One-sentence summary of the dominant market narrative |
| `sector_movers` | `list[SectorMover]` | Sectors with notable moves and brief reason |
| `macro_events` | `list[NewsItem]` | Macro-only items (Fed, CPI, GDP, etc.) |
| `watchlist_alerts` | `list[NewsItem]` | High-relevance items for the requested tickers |

**Tool parameters:**
- `tickers` — comma-separated watchlist symbols (e.g. `"NVDA, AAPL, TSLA"`)
- `sectors` — comma-separated sector names (e.g. `"Technology, Healthcare"`)
- `date` — ISO date string for the digest header (e.g. `"2026-03-21"`)
- `days_back` — lookback window in days (default 1)

> **Raises:** `RuntimeError` if LLM synthesis fails. The tool converts this to `"ERROR: …"` so the orchestrator can surface it gracefully.

In [ ]:
if not _HAS_KEYS:
    print("Skipping — requires TAVILY_API_KEY and OPENAI_API_KEY.")
else:
    result = agent.invoke(
        {"messages": [{"role": "user", "content":
            "Build a daily market digest for today (2026-03-21). "
            "Watchlist: NVDA, AAPL, TSLA. Sectors: Technology, Consumer Discretionary. "
            "Look back 1 day. Highlight the market theme and any watchlist alerts."}]},
        config=CONFIG,
    )
    print(get_reply(result))

## 13 — Multi-turn conversation <a id="13-multi-turn"></a>

Because `build_webai_agent` uses a `MemorySaver` checkpointer, the orchestrator remembers the full conversation history for a given `thread_id`. Follow-up questions can reference earlier results without restating context.

Key rules:
- Use the **same `thread_id`** across all invocations in a session.
- Use a **new `thread_id`** to start a fresh session with no memory of previous exchanges.
- Sub-agents are still **stateless** — the orchestrator threads context to them in each new `task` call.

In [ ]:
if not _HAS_KEYS:
    print("Skipping — requires TAVILY_API_KEY and OPENAI_API_KEY.")
else:
    # Turn 1 — seed with a ticker research
    mt_thread = str(uuid.uuid4())
    mt_config = {"configurable": {"thread_id": mt_thread}}

    r1 = agent.invoke(
        {"messages": [{"role": "user", "content": "Research AAPL (Apple) for the last 7 days."}]},
        config=mt_config,
    )
    print("=== Turn 1 ===")
    print(get_reply(r1))

In [ ]:
if not _HAS_KEYS:
    print("Skipping — requires TAVILY_API_KEY and OPENAI_API_KEY.")
else:
    # Turn 2 — follow-up referencing the previous result; no need to restate ticker
    r2 = agent.invoke(
        {"messages": [{"role": "user", "content":
            "Based on what you just found, what are the top 3 risk factors for AAPL "
            "and how do they relate to the current macro environment?"}]},
        config=mt_config,  # same thread_id — orchestrator remembers Turn 1
    )
    print("=== Turn 2 (follow-up) ===")
    print(get_reply(r2))

## 14 — End-to-end pattern <a id="14-end-to-end"></a>

A realistic morning-brief workflow that chains **all three sub-agents**:

1. **Macro context** — `fetch_macro_news` for broad market colour
2. **Portfolio research** — `research_portfolio` for per-stock snapshots
3. **Sector context** — `research_sector` for each unique sector in the watchlist
4. **Daily digest** — `fetch_daily_digest` to tie everything together

The orchestrator uses `write_todos` to plan and track each step, then delegates each to the right sub-agent, and synthesises a final brief.

> This single multi-step prompt exercises every sub-agent and tool in the system.

In [ ]:
MORNING_BRIEF_PROMPT = """
You are preparing a morning market brief. Please complete all of the following steps:

1. Fetch macro news for the last 2 days to set the broad market context.
2. Research this portfolio in parallel (last 7 days):
   - NVDA|NVIDIA (Semiconductors)
   - AAPL|Apple (Technology)
   - JNJ|Johnson & Johnson (Healthcare)
3. Research each unique sector once (no duplicate calls):
   - Semiconductors
   - Technology
   - Healthcare
4. Build a daily digest for today (2026-03-21) with:
   - Tickers: NVDA, AAPL, JNJ
   - Sectors: Semiconductors, Technology, Healthcare
   - days_back: 1
5. Combine all of the above into a single morning brief:
   - Macro summary (2-3 bullets)
   - Per-stock snapshot table: symbol | sentiment | confidence | key_catalyst
   - Sector health summary (1 sentence each)
   - Market theme from the digest
   - Top 3 watchlist alerts
""".strip()

print("Morning brief prompt defined")

In [ ]:
if not _HAS_KEYS:
    print("Skipping — requires TAVILY_API_KEY and OPENAI_API_KEY.")
else:
    e2e_config = {"configurable": {"thread_id": str(uuid.uuid4())}}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": MORNING_BRIEF_PROMPT}]},
        config=e2e_config,
    )
    print(get_reply(result))

## 15 — Error handling reference <a id="15-error-handling"></a>

### Factory-level errors (no API keys needed)

| Component | Guard condition | Error type |
|---|---|---|
| `build_webai_agent` | `TAVILY_API_KEY` missing (caught by `NewsResearcher.__init__`) | `ValueError` |
| `WebSearcher.search()` | provider client not initialised (key missing) | `ValueError` — surfaced as `"ERROR: …"` by tool |

### Tool-level error isolation

All 8 tools catch `Exception` and return `"ERROR: <message>"` — the orchestrator never crashes due to a single tool failure. The orchestrator's system prompt instructs it to relay errors to the user and suggest alternatives.

### Sub-agent errors

| Scenario | Behaviour |
|---|---|
| `research_ticker` on empty symbol | `ValueError` → `"ERROR: …"` string |
| `research_portfolio` — one ticker fails | Logged `WARNING`, excluded from result, rest returned |
| `fetch_daily_digest` — LLM synthesis fails | `RuntimeError` → `"ERROR: …"` string |
| Any tool — network / API failure | Exception caught → `"ERROR: …"` string |

### Fallback behaviour (not errors)

| Scenario | Behaviour |
|---|---|
| All results removed by domain filter | Falls back to unfiltered set, logs `WARNING` |
| Primary search provider fails | Retries with other provider (`fallback=True`) |
| Orchestrator sub-agent returns `"ERROR: …"` | Orchestrator surfaces to user gracefully |

In [ ]:
# Demonstrate tool-level error isolation without any API keys
# The tool wraps the real client, so we can simulate a provider failure
# by passing a mock that raises.

from unittest.mock import MagicMock, patch
from agents.tools.web_search_tools import make_web_search_tools
from webai import WebSearcher

failing_searcher = MagicMock()
failing_searcher.search.side_effect = ConnectionError("Tavily unreachable")
failing_tools = make_web_search_tools(failing_searcher)

# Invoke the tool directly — confirm it returns "ERROR: …" not raises
err_result = failing_tools[0].invoke({"query": "test query"})
print(f"Tool error isolation : {err_result}")

In [ ]:
# Demonstrate that NewsResearcher raises ValueError when TAVILY_API_KEY is missing
import os as _os

_saved_tavily = _os.environ.pop("TAVILY_API_KEY", None)
try:
    from langchain_openai import ChatOpenAI as _ChatOpenAI
    from webai import NewsResearcher as _NewsResearcher
    _m = _ChatOpenAI(model="gpt-4o-mini", api_key=_os.environ.get("OPENAI_API_KEY", "sk-dummy"))
    _NewsResearcher(model=_m)
except ValueError as e:
    print(f"NewsResearcher(no Tavily key) -> ValueError: {e}")
finally:
    if _saved_tavily:
        _os.environ["TAVILY_API_KEY"] = _saved_tavily